# 1D CNN Training for ASL-Sign-50

1D convolutional neural network for classifying 50 ASL signs from the Google Kaggle ASL Signs dataset. The data is using the preprocessed data from our preprocessing files.

Input shape: (30, 274) sequences where 274 is features per frame (positional + velocity landmarks).

Conv1D filters slide along the time axis, learning motion patterns across the 30 frames. It is a 1D CNN becuase the input is a time series, not an image. We already have the feature landmarks from mediapipe and the feature engineering. Sliding along the time axis will allow the model to learn motion and distinguish between signs.

**Pipeline:**
1. **Load preprocessed data from: Processed_ASL_Data:**
- Dev set (dev.npz): 18 participants used for training and cross-validation
- Test set (test.npz): 3 held-out participants never seen during training
- Pre-computed fold indices (cv_folds.npz): shared across RF, CNN, and LSTM for fair comparison
- Classes (classes.npy): 50 ASL sign labels
- Input shape: (30, 274) — 30 frames × 274 features (137 positional + 137 velocity landmarks)

2. **Focused grid search with participant senstive 5-fold CV for hyperparameter tuning:**
- 5 hyperparameters searched: n_conv_layers, n_filters, kernel_size, learning rate, dropout
- n testing values per parameter, values changed after analyzing the data each run of this notebook for tunning
- Each configuration trained 5 times (once per fold) -> 160 total training runs
- Folds ensure no participant appears in both train and validation splits
- Selection criterion: mean macro F1 across all 5 folds
- Early stopping if the accuracy has not improved in multiple epochs
- ReduceLROnPlateau: watches validation loss and changes the learning rate if there hasn't been improvment in the last few epochs, to try to escape any "plateaus" before early stopping takes effect

3. **Final model trained on full dev set (train and validation data), evaluated on holdout test set:**
- Best hyperparameter config from grid search used to build final model
- Trained on all 18 dev participants combined for FINAL_EPOCHS
- ReduceLROnPlateau learning rate schedule: halves LR when training loss stalls for patience=n epochs
- Test set touched exactly once at the end

4. **Per-class accuracy, macro F1, and confusion matrix:**
- Overall accuracy and macro F1 reported on holdout test set
- Per-class accuracy breakdown for all 50 signs
- Confusion matrix and per-class bar chart saved as figures
- All results saved to cnn1d_results.pkl for comparison notebook

**Speed optimizations:**
- Mixed precision (float16 compute) — ~2x faster on V100/A100 Tensor Cores for bridges2
- pipeline with prefetching — GPU never waits for CPU to prepare batches, they are prepared ahead of time
- Larger batch size maximizes GPU utilization per step

## Imports and Configuration

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # suppress INFO and WARNING messages

import time # measure how long training takes
import itertools # used for getting all combinations of hyperparameters
import pickle # save training history and results

import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision # mixed precision for faster training on bridges2 GPUs

from tqdm import tqdm # progress bars for training loops
from tqdm.keras import TqdmCallback # progress bars for keras model.fit() training loops

from sklearn.metrics import f1_score, confusion_matrix, classification_report
# the F1 score is calculated by precision (true postives / predicted positives) and recall (true positives / actual positives)
# the macro F1 score is the mean F1 score across all classes, treating them equally regardless of class imbalance
# the confusion matrix shows the counts of true positives, false positives, true negatives, and false negatives for each class
# the classification report includes precision, recall, F1 score, and support (number of true instances) for each class

# visualization libraries
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# ── GPU detection ─────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        # allow memory growth so TF doesn't claim all VRAM at startup
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU(s) available: {[g.name for g in gpus]}")
else:
    print("No GPU found — running on CPU")

# ── Mixed Precision ───────────────────────────────────────────────────────────
# Uses float16 for compute (matrix multiplications in Conv1D, Dense layers)
# and float32 for weight storage -> faster on V100/A100 Tensor Cores with no meaningful accuracy loss
mixed_precision.set_global_policy('mixed_float16')
print(f"Compute dtype: {mixed_precision.global_policy().compute_dtype}")
print(f"Variable dtype: {mixed_precision.global_policy().variable_dtype}")

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = './data/Processed_ASL_Data'
RESULTS_DIR = './CNN_Results2'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 80
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"\nAll imports successful")
print(f"TensorFlow: {tf.__version__}")

## Load Preprocessed Data

Load the same splits used by all three models (RF, CNN, LSTM) for a fair comparison that was created by the preprocessing notebook.
- `dev.npz`: training + validation data (18 participants), used for CV and final training
- `test.npz`: holdout test set (3 unseen participants), never touched during training
- `cv_folds.npz`: pre-computed fold indices so all models see the exact same splits

In [ ]:
# Load dev set
dev        = np.load(os.path.join(DATA_DIR, 'dev.npz'))
X_dev      = dev['X'].astype(np.float32)   # (N_dev, 30, F)
y_dev      = dev['y'].astype(np.int32)     # (N_dev,) integer class labels
groups_dev = dev['groups']                 # (N_dev,) participant IDs

# Load holdout test set
test   = np.load(os.path.join(DATA_DIR, 'test.npz'))
X_test = test['X'].astype(np.float32)     # (N_test, 30, F)
y_test = test['y'].astype(np.int32)       # (N_test,)

# Load pre-computed fold indices
folds_npz    = np.load(os.path.join(DATA_DIR, 'cv_folds.npz'))
N_FOLDS      = sum(1 for k in folds_npz if k.endswith('_train'))
fold_indices = [(folds_npz[f'fold_{i}_train'], folds_npz[f'fold_{i}_val']) for i in range(N_FOLDS)]

# Load class names
classes   = np.load(os.path.join(DATA_DIR, 'classes.npy'), allow_pickle=True)
N_CLASSES = len(classes)

N_FRAMES   = X_dev.shape[1]   # 30
N_FEATURES = X_dev.shape[2]   # 274 (137 positional + 137 velocity)

print(f"\nDev set:   X={X_dev.shape}, y={y_dev.shape} | {len(np.unique(groups_dev))} participants")
print(f"Test set:  X={X_test.shape}, y={y_test.shape}")
print(f"Classes: {N_CLASSES} | Frames: {N_FRAMES} | Features/frame: {N_FEATURES}")
print(f"CV folds: {N_FOLDS}")

## tf.data Pipeline

Instead of feeding numpy arrays directly to the model fit, we built a tf.data.Dataset pipeline to prepare the batches ahead of time.
This runs data preparation (batching, shuffling) on the CPU in parallel with GPU training,
so the GPU is never sitting idle waiting for the next batch to be ready.

prefetch(AUTOTUNE): tells TensorFlow to prepare the next batch while the current one is training, this is the key optimization that eliminates CPU/GPU idle time and makes the program run faster.

In [ ]:
def make_dataset(X, y, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y)) # create tf.data.Dataset from numpy arrays
    if shuffle:
        # shuffle the full dataset each epoch
        ds = ds.shuffle(buffer_size=len(X), seed=RANDOM_SEED)
    ds = ds.batch(batch_size) # batch the data
    ds = ds.prefetch(tf.data.AUTOTUNE)  # prepare next batch while GPU trains on current
    return ds # returns a tf.data.Dataset yielding batches of (X, y)

# ── Quick test ────────────────────────────────────────────────────────────────
test_ds = make_dataset(X_dev[:100], y_dev[:100], batch_size=32, shuffle=True)
for batch_X, batch_y in test_ds.take(1):
    print(f"Batch X shape: {batch_X.shape}")
    print(f"Batch y shape: {batch_y.shape}")

## Model Definition

Keras functional API builds the CNN from a config dict of hyperparameters that is later defined.
Input shape (30, 274) Conv1D slides filters along the time axis (30 frames), treating the 274 features as channels.

Architecture per convolution block: Conv1D -> BatchNorm -> ReLU -> MaxPool -> Dropout
Filter count doubles each block to capture increasingly abstract time dependent patterns.
GlobalAveragePooling1D collapses the time dimension into a single vector per sample, before the dense classifier head(pretty much Flatten) reduces overfitting on our dataset.


In [ ]:
def build_model(config):
    inputs = keras.Input(shape=(N_FRAMES, N_FEATURES), name='sequence_input')  # (30, 274)
    x      = inputs

    for i in range(config['n_conv_layers']): # run n_conv_layers times
        n_filters = config['n_filters'] * (2 ** i)  # double filters each block
        x = layers.Conv1D(
            filters     = n_filters,
            kernel_size = config['kernel_size'],
            padding     = 'same',    # keep temporal dimension unchanged after conv
            activation  = None,
            name        = f'conv_{i}'
        )(x)
        x = layers.BatchNormalization(name=f'bn_{i}')(x)           # normalize batch before activation
        x = layers.ReLU(name=f'relu_{i}')(x)                       # applies the rectified linear activation function
        x = layers.MaxPooling1D(pool_size=2, name=f'pool_{i}')(x)  # apply max pooling
        x = layers.Dropout(config['dropout'], name=f'drop_{i}')(x) # dropout for regularization

    # GlobalAveragePooling collapses remaining time steps into one vector per sample
    # more regularizing than Flatten, avoids exploding param count at different depths
    x = layers.GlobalAveragePooling1D(name='gap')(x)

    x = layers.Dense(128, activation='relu', name='dense_head')(x) # add a dense (fully connected) layer before the output
    x = layers.Dropout(config['dropout'], name='drop_head')(x)     # dropout before output layer as well

    # softmax activation for multi-class classification, outputs class probabilities 0-1
    outputs = layers.Dense(N_CLASSES, activation='softmax', dtype='float32', name='output')(x)

    # packages the model architecture and training configuration together
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=config['lr']),
        loss      = 'sparse_categorical_crossentropy',  # labels are integers, not one-hot
        metrics   = ['accuracy']
    )
    
    return model


# ── Sanity check: verify output shape and print parameter count ───────────────
test_config = {'n_conv_layers': 3, 'n_filters': 64, 'kernel_size': 3, 'dropout': 0.3, 'lr': 1e-3}
test_model  = build_model(test_config)
test_model.summary()
del test_model
keras.backend.clear_session()

## Hyperparameter Search via 5-Fold Group-Aware CV

Focused grid search: we use a small set of hyperparameter configs rather than an exhaustive sweep. We then tunned the hyperparameters in testing batches to optimize the model.
Each configuration is trained with early stopping per fold; the best val_loss weights are restored.
Selection criterion: mean validation macro F1 across all 5 folds.

The same fold indices used by RF and LSTM ensure a fair cross-model comparison.

**Search space:**
- `n_conv_layers` — network depth (2 or 3 blocks)
- `n_filters` — base filter count (doubles each layer)
- `kernel_size` — temporal receptive field per filter
- `lr` — Adam learning rate
- `dropout` — regularization strength

Batch size fixed at 256 — maximizes GPU utilization on V100/A100 with 32GB VRAM.

In [ ]:
# ── Focused hyperparameter grid ───────────────────────────────────────────────
PARAM_GRID = {
    'n_conv_layers': [3],           # 3 was best for most runs
    'n_filters'    : [128, 256],    # similar results for 128 vs 256 filters
    'kernel_size'  : [4, 5, 6],     # 5 consistently the best, keeping 4 and 6 for comparison
    'lr'           : [1e-3, 2e-3],  # 2e-3 was best mostly, keeping 1e-3 for comparison 
    'dropout'      : [0.0, 0.1],    # 0.1 dominated, trying no dropout at all too
}
BATCH_SIZE  = 256  # large batch — maximizes GPU utilization, fits in V100 32GB VRAM
CV_EPOCHS   = 25   # max epochs per fold; early stopping will usually kick in earlier
ES_PATIENCE = 6    # stop if val_loss hasn't improved in 6 consecutive epochs

keys   = list(PARAM_GRID.keys()) # ['n_conv_layers', 'n_filters', 'kernel_size', 'lr', 'dropout']
combos = list(itertools.product(*PARAM_GRID.values())) # get all combinations of hyperparameters from the grid
print(f"Total configurations : {len(combos)}")
print(f"Total training runs  : {len(combos)} configs * {N_FOLDS} folds = {len(combos)*N_FOLDS}")
print(f"Max epochs per run   : {CV_EPOCHS} (early stopping patience={ES_PATIENCE})")
print(f"Batch size           : {BATCH_SIZE}")

In [ ]:
# ── Grid search training ───────────────────────────────────────────────────────
cv_results = []  # list of dicts: {params, fold_f1s, mean_f1, std_f1}
t_start    = time.time()

# outer loop: configs
config_bar = tqdm(enumerate(combos), total=len(combos), desc="Grid Search", unit="config")

for combo_idx, values in config_bar:
    config   = dict(zip(keys, values))   # e.g. {'n_conv_layers': 3, 'n_filters': 64, ...}
    fold_f1s = []

    # inner loop: folds
    fold_bar = tqdm(enumerate(fold_indices), total=N_FOLDS, desc=f"  Config {combo_idx+1}/{len(combos)} folds", leave=False, unit="fold")

    for fold_idx, (tr_idx, va_idx) in fold_bar:
        # Data split 
        X_tr, y_tr = X_dev[tr_idx], y_dev[tr_idx]
        X_va, y_va = X_dev[va_idx], y_dev[va_idx]

        # build tf.data pipelines — shuffle training, no shuffle for validation
        train_ds = make_dataset(X_tr, y_tr, BATCH_SIZE, shuffle=True)
        val_ds   = make_dataset(X_va, y_va, BATCH_SIZE, shuffle=False)

        # Build a fresh model for each fold 
        keras.backend.clear_session()  # release GPU memory between folds
        tf.random.set_seed(RANDOM_SEED)
        model = build_model(config)

        # Callbacks 
        callbacks = [ # stop training early if validation loss does not improve for ES_PATIENCE epochs, and reduce LR if val_loss plateaus for 3 epochs
            keras.callbacks.EarlyStopping(
                monitor              = 'val_loss',
                patience             = ES_PATIENCE,
                restore_best_weights = True,  # revert to best epoch weights on stop
                verbose              = 0
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor  = 'val_loss',
                factor   = 0.5,   # halve LR when triggered
                patience = 3,     # trigger after 3 stagnant epochs
                min_lr   = 1e-6,
                verbose  = 0
            )
        ]

        # Train 
        model.fit(
            train_ds,
            validation_data = val_ds,
            epochs          = CV_EPOCHS,
            callbacks       = callbacks,
            verbose         = 0   # suppress per-epoch output during grid search
        )

        # Score this fold on macro F1 (mean F1 across classes)
        y_pred  = model.predict(val_ds, verbose=0).argmax(axis=1)
        fold_f1 = f1_score(y_va, y_pred, average='macro')
        fold_f1s.append(fold_f1)
        fold_bar.set_postfix(fold_f1=f'{fold_f1:.4f}')

    mean_f1 = np.mean(fold_f1s)
    std_f1  = np.std(fold_f1s)
    cv_results.append({'params': config, 'fold_f1s': fold_f1s, 'mean_f1': mean_f1, 'std_f1': std_f1}) # store results for this config

    elapsed = (time.time() - t_start) / 60
    config_bar.set_postfix(best_f1=f'{max(r["mean_f1"] for r in cv_results):.4f}', elapsed=f'{elapsed:.1f}m') # show best F1 so far and elapsed time

# ── Select best configuration ─────────────────────────────────────────────────
best_result = max(cv_results, key=lambda r: r['mean_f1']) # get config with highest mean F1 across folds
best_params = best_result['params'] # the hyperparameters for the best config
print(f"\nBest config      : {best_params}")
print(f"Best CV macro F1 : {best_result['mean_f1']:.4f} ± {best_result['std_f1']:.4f}")

In [ ]:
# CV results summary — top 10 configs 
sorted_results = sorted(cv_results, key=lambda r: r['mean_f1'], reverse=True) 

print(f"{'Rank':<5} {'mean_F1':<10} {'std_F1':<9} {'layers':<8} {'filters':<9}"
      f"{'kernel':<8} {'lr':<9} {'dropout'}")
print('-' * 70)
for rank, r in enumerate(sorted_results[:10], 1):
    p = r['params']
    print(f"{rank:<5} {r['mean_f1']:<10.4f} {r['std_f1']:<9.4f} "
          f"{p['n_conv_layers']:<8} {p['n_filters']:<9} {p['kernel_size']:<8} "
          f"{p['lr']:<9.0e} {p['dropout']}")

## Final Model Training

Train the hyperparameter best configuration on the full dev set (all 18 participants combined).

More epochs are used here since we now have more data to learn from. No validation split this time, we train on everything because we already got our best hyperparameters.

ReduceLROnPlateau monitors training loss and halves the learning rate if it stagnates for 5 epochs, allowing the model to keep improving rather than stopping early.
The holdout test set is only touched once at the very end for the final evaluation.

In [ ]:
FINAL_EPOCHS = 100 # train for more epochs on the full dev set with the best hyperparameters

keras.backend.clear_session() # release GPU memory before final training
tf.random.set_seed(RANDOM_SEED)
final_model = build_model(best_params)

final_model.compile( # recompile with best hyperparametrs
    optimizer = keras.optimizers.Adam(learning_rate=best_params['lr']),
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy']
)

print(f"Final model | trainable params: {final_model.count_params():,}")
print(f"Training for {FINAL_EPOCHS} epochs on {len(X_dev)} samples...\n")

t0 = time.time()
# build tf.data pipeline for final training — shuffle on since we train on full dev set
final_train_ds = make_dataset(X_dev, y_dev, BATCH_SIZE, shuffle=True)

history = final_model.fit(
    final_train_ds, # train on full dev set with best hyperparameters
    epochs    = FINAL_EPOCHS, # early stopping will prevent overfitting
    callbacks = [   
        TqdmCallback(verbose=1),  # tqdm progress bar per epoch
        keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)
    ],
    verbose   = 0   # suppress default Keras output, tqdm handles it
)

print(f"\nTraining complete in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── Training curve ────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4)) # create side-by-side subplots for loss and accuracy

ax1.plot(history.history['loss'], label='train')
if 'val_loss' in history.history:
    ax1.plot(history.history['val_loss'], label='val')
    ax1.legend()
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['accuracy'], label='train')
if 'val_accuracy' in history.history:
    ax2.plot(history.history['val_accuracy'], label='val')
    ax2.legend()
ax2.set_title('Training Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curve.png'), dpi=150, bbox_inches='tight')
plt.close()  # use close() instead of show() — works in both notebook and sbatch

## Test Set Evaluation

Final evaluation on the 3 held-out participants run only once at the end of our training.
Metrics: overall accuracy, macro F1, per-class accuracy, confusion matrix.

In [ ]:
# ── Run inference on holdout test set ─────────────────────────────────────────
test_ds      = make_dataset(X_test, y_test, batch_size=512, shuffle=False)
y_pred_probs = final_model.predict(test_ds, verbose=0)
y_pred       = y_pred_probs.argmax(axis=1)

_, test_acc   = final_model.evaluate(test_ds, verbose=0)
macro_f1      = f1_score(y_test, y_pred, average='macro')
cm            = confusion_matrix(y_test, y_pred)
per_class_acc = {classes[i]: cm[i, i] / cm[i].sum() for i in range(N_CLASSES)}

print(f"=== Holdout Test Set Results ===")
print(f"  Overall Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Macro F1 Score   : {macro_f1:.4f}")
print(f"\nPer-class accuracy (sorted high to low):")
for sign, acc in sorted(per_class_acc.items(), key=lambda x: x[1], reverse=True):
    print(f"  {sign:<15} {acc:.4f}")

In [ ]:
# ── Full sklearn classification report ────────────────────────────────────────
print(classification_report(y_test, y_pred, target_names=classes))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=classes, yticklabels=classes,
    linewidths=0.4, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'1D CNN — Confusion Matrix (Test Set) | Macro F1={macro_f1:.4f}', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
# ── Per-class accuracy bar chart ──────────────────────────────────────────────
sorted_signs = sorted(per_class_acc.items(), key=lambda x: x[1], reverse=True)
signs, accs  = zip(*sorted_signs)

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(signs, accs, color='steelblue', edgecolor='white')
ax.axhline(y=test_acc, color='red', linestyle='--', label=f'Overall acc = {test_acc:.3f}')
ax.set_xlabel('Sign')
ax.set_ylabel('Accuracy')
ax.set_title('1D CNN — Per-Class Accuracy on Holdout Test Set')
ax.set_ylim(0, 1.05)
plt.xticks(rotation=45, ha='right', fontsize=8)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'per_class_accuracy.png'), dpi=150, bbox_inches='tight')
plt.close()

## Save Model and Results

In [ ]:
# Save Keras model in native format (architecture + weights + optimizer state)
final_model.save(os.path.join(RESULTS_DIR, 'cnn1d_final_model.keras'))

# Save all CV results and summary metrics as a dict for the comparison notebook
results_summary = {
    'best_params'   : best_params,
    'cv_mean_f1'    : best_result['mean_f1'],
    'cv_std_f1'     : best_result['std_f1'],
    'test_accuracy' : float(test_acc),
    'test_macro_f1' : float(macro_f1),
    'per_class_acc' : per_class_acc,
    'cv_results'    : cv_results,
}
with open(os.path.join(RESULTS_DIR, 'cnn1d_results.pkl'), 'wb') as f:
    pickle.dump(results_summary, f)

print(f"=== Saved to {RESULTS_DIR}/ ===")
for fname in sorted(os.listdir(RESULTS_DIR)):
    size = os.path.getsize(os.path.join(RESULTS_DIR, fname)) / 1024
    print(f"  {fname}: {size:.1f} KB")

print(f"\n=== Final Summary ===")
print(f"Best hyperparameters : {best_params}")
print(f"CV macro F1          : {best_result['mean_f1']:.4f} ± {best_result['std_f1']:.4f}")
print(f"Test accuracy        : {test_acc*100:.2f}%")
print(f"Test macro F1        : {macro_f1:.4f}")